# Response Hyperalignment (RHA)

Response Hyperalignment (RHA) aligns subject-specific response patterns into a shared representational space. This notebook documents the same RHA stages for script-based use and for the graphical interface: searchlight preparation, common-space construction, transformation matrix calculation, and applying transformation matrices.

The examples assume that response data have already been prepared as `.npy` arrays with shape `timepoints x features` under the HA work directory. Surface folders usually use `hemi-L` and `hemi-R`. CIFTI subcortical folders use `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`. NIfTI ROI folders use `volume-<roi_name>`.


## Scripts Pipeline

The script examples estimate common spaces and transformation matrices from `run="01"`, then apply the transformations to `run="02"`. Replace all example paths, subject IDs, task names, ROI names, and file filters with values matching your dataset.

**Required script inputs**

| Item | Meaning |
|---|---|
| `work_dir` | Root HA work directory. It contains subject folders, `logs`, `masks`, common-space outputs, transform outputs, and aligned outputs. |
| `json_path` | Processing-record JSON path, usually `work_dir / "logs" / "prep_log.json"`. |
| `sub_list` | Subject folder names, such as `sub-rid000005`. Use the same subjects for common-space construction, transformation calculation, and alignment unless your design requires a stage-specific subject set. |
| Prepared response arrays | `.npy` files selected through BIDS-like filename entities. Common filters include `run`, `ses`, `set`, `space`, `hemi`, `roi`, `res`, `desc`, and `task`. |
| Searchlights | `sls` and `dists` objects whose indices match the feature dimension of each prepared structure. |
| Task names | Labels written into output filenames. Use distinct task names for cortical, subcortical, and ROI workflows when running them in the same work directory. |


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path

import joblib
import nibabel as nib
import numpy as np
import neuroboros as nb

from fmriha import gensls, ha


In [ ]:
work_dir = Path("/path/to/ha_workdir")
json_path = work_dir / "logs" / "prep_log.json"
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

fit_file_filters = {"run": "01"}
align_source_filter = {"run": "02"}
common_space_filter = {"alg": "procrustes"}


### (1) Searchlight Preparation

Searchlights define the local neighborhoods used by RHA. The script functions receive precomputed `sls` and `dists`; searchlight creation is handled in this preparation section.

**Cortical**

For cortical surface data, `neuroboros.sls` is the standard helper used by the GUI runner and by the examples here.

Parameters of `nb.sls`:

| Parameter | Default | Meaning |
|---|---|---|
| `lr` | Required | Hemisphere selector, such as `"l"` or `"r"`. |
| `radius` | Required | Surface searchlight radius in millimeters. |
| `space` | `"onavg-ico32"` | Surface space used for searchlight vertices. |
| `center_space` | `None` | Optional surface space used for searchlight centers. With `None`, centers use `space`. |
| `mask` | `True` | Cortical mask for searchlight vertices. It can be a boolean array or a boolean control value. |
| `center_mask` | `None` | Cortical mask for searchlight centers. With `None`, it follows `mask`. |
| `return_dists` | `False` | If `True`, returns `(sls, dists)`. If `False`, returns only `sls`. |


In [ ]:
radius_surf = 20
surface_space = "onavg-ico32"

surface_tmp = {
    lr: nb.sls(
        lr,
        radius_surf,
        surface_space,
        mask=True,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf = {lr: surface_tmp[lr][0] for lr in surface_tmp}
dists_surf = {lr: surface_tmp[lr][1] for lr in surface_tmp}


`gensls.sls_update` is optional. Use this helper when searchlights were generated in full-surface indexing and the prepared response arrays use a custom masked indexing scheme.

Parameters of `gensls.sls_update`:

| Parameter | Default | Meaning |
|---|---|---|
| `sls` | Required | Dictionary of searchlight index lists. Keys are usually `"l"` and `"r"`. |
| `dists` | `None` | Dictionary of distance arrays corresponding to `sls`. With `None`, only indices are updated. |
| `medial_wall` | `None` | Dictionary of boolean masks in full-surface indexing. `True` marks vertices removed from the prepared response arrays. |
| `lr` | `None` | Hemispheres to process, for example `"lr"` or `["l", "r"]`. |
| `center_type` | `"first"` | Center-selection rule. `"first"` uses the first vertex in each searchlight. `"idx"` uses the searchlight list position. |


In [ ]:
medial_wall = joblib.load("/path/to/custom_medial_wall.pkl")

full_surface_tmp = {
    lr: nb.sls(
        lr,
        radius_surf,
        surface_space,
        mask=False,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf_full = {lr: full_surface_tmp[lr][0] for lr in full_surface_tmp}
dists_surf_full = {lr: full_surface_tmp[lr][1] for lr in full_surface_tmp}

sls_surf, dists_surf = gensls.sls_update(
    sls_surf_full,
    dists=dists_surf_full,
    medial_wall=medial_wall,
    lr="lr",
    center_type="first",
)


**Subcortical**

For CIFTI subcortical RHA, derive dense left, right, and brain-stem masks from a reference CIFTI file, then generate a volume searchlight inside each mask.

Parameters of `gensls.downsample_cifti_subcortical`:

| Parameter | Default | Meaning |
|---|---|---|
| `cifti_file` | Required | Reference CIFTI file used to read subcortical voxel coordinates and volume dimensions. |
| `N` | Required | Target number of sampled voxels for downsampled modes. Use `None` with `sls_type="seed"` to keep all available subcortical voxels. |
| `mask3d_out` | `None` | Output directory for saving masks. With `None`, masks are returned in memory. |
| `sls_type` | `"subcortex"` | Sampling mode label. This workflow uses `"seed"` for dense searchlight centers. |
| `voxel_sizes` | `None` | Optional voxel spacing used for spatial sampling. With `None`, voxel-index coordinates are used. |
| `drop_all_zero_columns` | `True` | Removes CIFTI subcortical columns that are zero across all time points before downsampling. |
| `whole_mask` | `False` | When saving masks, controls whether the whole subcortical mask is saved. |
| `hemi_mask` | `True` | When saving masks, controls whether separate left, right, and brain-stem masks are saved. |

Parameters of `gensls.generate_searchlight_vol`:

| Parameter | Default | Meaning |
|---|---|---|
| `mask_dense` | Required | Dense 3D mask defining voxels allowed inside searchlights. It can be a NumPy array or a NIfTI file path. |
| `mask_center` | `None` | Optional 3D mask defining searchlight centers. With `None`, every non-zero voxel in `mask_dense` can be a center. |
| `radius` | `2` | Volume searchlight radius in voxel units. This notebook uses `radius_vol = 4`. |
| `threshold` | `0` | Minimum fraction of in-mask voxels required for a candidate searchlight to be kept. This notebook uses `0.2`. |
| `njobs` | `0` | Number of parallel jobs. `0` runs sequentially. |
| `verbose` | `0` | Joblib verbosity level. |

The returned volume-searchlight dictionary contains `sls_coord`, `sls_idx`, and `dists`. RHA uses `sls_idx` and `dists`.


In [ ]:
reference_cifti = "/path/to/reference_run01.dtseries.nii"
radius_vol = 4

mask_dense_subcortical = {
    lr: gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=None,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

sls_info_subcortical = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_subcortical[lr],
        mask_center=None,
        radius=radius_vol,
        threshold=0.2,
        njobs=10,
        verbose=5,
    )
    for lr in ["l", "r", "brain_stem"]
}

sls_subcortical = {
    f"subcortical-{lr}": sls_info_subcortical[lr]["sls_idx"]
    for lr in ["l", "r", "brain_stem"]
}

dists_subcortical = {
    f"subcortical-{lr}": sls_info_subcortical[lr]["dists"]
    for lr in ["l", "r", "brain_stem"]
}


**Volume ROI**

For an ROI defined by a NIfTI mask, use the non-zero mask voxels as the dense searchlight mask. The ROI name becomes part of the structure folder name, such as `volume-mPFC`. The same `gensls.generate_searchlight_vol` parameter table above applies to this ROI workflow.


In [ ]:
roi_name = "mPFC"
nifti_mask_path = "/path/to/mPFC_mask.nii.gz"

mask_dense_roi = nib.load(nifti_mask_path).get_fdata().astype(bool)

sls_info_roi = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=None,
    radius=radius_vol,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi = {f"volume-{roi_name}": sls_info_roi["sls_idx"]}
dists_roi = {f"volume-{roi_name}": sls_info_roi["dists"]}


### (2) Common Space Construction

`ha.cspace_sls` constructs a whole-structure common representational space by computing local searchlight common spaces and aggregating them over the target feature axis.

Parameters of `ha.cspace_sls`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. Input files are searched under `work_dir / sub / step / modality / structure_name`. |
| `sls` | Required | Searchlight indices. Use a list for one structure or a dictionary keyed by hemisphere or volume structure. |
| `dists` | Required | Distance arrays corresponding to `sls`. Required when `weight=True`. |
| `radius` | Required | Searchlight radius used to compute distance weights. Use the same radius used during searchlight preparation. |
| `sub_list` | Required | Subject folder names included in the common-space estimate. |
| `task_name` | `"rha1"` | Name written into the common-space output filename as `task-<task_name>`. |
| `cspace_kind` | `"procrustes"` | Common-space algorithm. The GUI exposes `"procrustes"` and `"pca"`. |
| `common_topography` | `False` | If `True`, maps PCA-like common components back into feature topography. Use `False` for Procrustes RHA. |
| `weight` | `True` | Aggregates overlapping searchlights with distance-based weights. |
| `demean` | `True` | Demeans data before common-template computation. |
| `start_sub` | `0` | Reference subject for Procrustes common-space initialization. Use an integer subject index or `"mean"`. |
| `chunk_size` | `2000` | Number of searchlights processed per chunk. Larger chunks reduce scheduling overhead and require more memory. |
| `njobs` | `5` | Number of joblib workers for the local workflow. |
| `step` | `"resample"` | Prepared-data step folder, usually `"resample"` or `"combine"`. |
| `modality` | `"response"` | Data modality folder, usually `"response"` for RHA. |
| `structure_name` | `"hemi-L"` | Structure folder or list of folders to process, such as `"hemi-L"`, `"hemi-R"`, `"volume-subcortical-L"`, or `"volume-mPFC"`. |
| `dtype` | `np.float32` | Numeric dtype for saved common-space arrays. |
| `scope` | `"sls"` | Output naming label for the searchlight scope. If weighted, filenames use `scope-w<scope>`. |
| `json_path` | `"prep_log.json"` | JSON log path updated after common-space construction. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `dask` | `False` | Standard local execution setting for this notebook. |
| `cluster` | `None` | Cluster object paired with `dask`; keep `None` for the standard local workflow. |
| `**file_filters` | Empty | BIDS-like filename filters used to select source `.npy` files, such as `run="01"`, `set="train"`, or `desc="bold-zscore"`. |

Outputs are saved under `work_dir / "common_space" / modality / structure_name`.

**Cortical**

Use the cortical searchlights prepared for `hemi-L` and `hemi-R`.


In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    task_name="rhaPRcortical",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=5,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_cspace_cortical",
    **fit_file_filters,
)


**Subcortical**

Use the CIFTI-derived subcortical searchlights for the left subcortex, right subcortex, and brain stem.


In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="rhaPRsubcortical",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=5,
    step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_cspace_subcortical",
    **fit_file_filters,
)


**Volume ROI**

Use the ROI searchlights and match `structure_name` to the ROI folder created during data preparation.


In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="rhaPRroi",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=5,
    step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_cspace_roi",
    **fit_file_filters,
)


### (3) Transformation Matrix Calculation

`ha.xform_sls_con` calculates subject-specific sparse transformation matrices by aligning each subject's response data to an existing common-space file created by `ha.cspace_sls`.

Parameters of `ha.xform_sls_con`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sls` | Required | Searchlight indices matching both the prepared source data and the common-space feature dimension. |
| `dists` | Required | Distance arrays corresponding to `sls`. Required when `weight=True`. |
| `radius` | Required | Searchlight radius used to compute aggregation weights. |
| `sub_list` | Required | Subject folder names for transformation matrix calculation. |
| `source_step` | `"resample"` | Prepared-data step folder containing the subject data to align. |
| `modality` | `"response"` | Modality folder for source data and transform outputs. |
| `structure_name` | `["hemi-L", "hemi-R"]` | Structure folder or list of folders to process. |
| `task_name` | `"rha1"` | Common-space task name to select and transform task name written to output files. |
| `cspace_desc` | `None` | Optional BIDS-like filters for selecting the common-space file, such as `{"alg": "procrustes"}`. |
| `reflection` | `True` | Allows reflection in the Procrustes transform. This is the usual RHA setting. |
| `scaling` | `False` | Allows scaling in the Procrustes transform. `False` is typical for RHA. |
| `weight` | `True` | Aggregates local transformation matrices with distance-based weights. |
| `chunk_size` | `2000` | Number of searchlights processed per chunk. |
| `check_completeness` | `False` | Compatibility argument retained by the API. |
| `dtype` | `np.float32` | Numeric dtype used while building and saving sparse transformation matrices. |
| `njobs` | `5` | Number of joblib workers. |
| `scope` | `"sls"` | Output naming label for the transformation scope. If weighted, filenames use `scope-w<scope>`. |
| `json_path` | `"prep_log.json"` | JSON log path updated after transformation matrix calculation. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `**file_filters` | Empty | BIDS-like filename filters used to select source `.npy` files, such as `run="01"`, `set="train"`, or `desc="bold-zscore"`. |

Outputs are saved under `work_dir / sub / "xforms" / modality / structure_name` as sparse `.npz` matrices.

**Cortical**

Use the same cortical searchlights and task name used for the cortical common space.


In [ ]:
ha.xform_sls_con(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    task_name="rhaPRcortical",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=5,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_xform_cortical",
    **fit_file_filters,
)


**Subcortical**

Use the subcortical common-space task name and the CIFTI-derived subcortical searchlights.


In [ ]:
ha.xform_sls_con(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="rhaPRsubcortical",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=5,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_xform_subcortical",
    **fit_file_filters,
)


**Volume ROI**

Use the ROI common-space task name and the ROI searchlights.


In [ ]:
ha.xform_sls_con(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    task_name="rhaPRroi",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=5,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="rha_xform_roi",
    **fit_file_filters,
)


### (4) Apply Transformation Matrix

`ha.align_pipe` applies each subject's sparse transformation matrix to selected response data and saves aligned `.npy` arrays under each subject's `align` folder. The aligned matrix is computed as `raw_data @ xform`.

Parameters of `ha.align_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names to align. |
| `source_step` | Required | Step folder containing the response data to align, usually `"resample"` or `"combine"`. |
| `source_modality` | Required | Modality folder containing the response data to align, usually `"response"`. |
| `source_structure_name` | Required | Structure folder containing the response data to align. |
| `source_name_filter` | Required | Dictionary of BIDS-like filters used to select exactly one source `.npy` file per subject, such as `{"run": "02"}`. |
| `xform_modality` | Required | Modality folder under each subject's `xforms` directory, usually `"response"`. |
| `xform_structure_name` | Required | Transform structure folder. Use `None` to use `source_structure_name`. |
| `xform_name_filter` | Required | Dictionary of BIDS-like filters used to select exactly one transform `.npz` file per subject, such as `{"task": "rhaPRcortical"}`. |
| `njobs` | `5` | Number of parallel workers. The standard path uses joblib threading. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Numeric dtype for saved aligned arrays. |
| `json_path` | `"prep_log.json"` | JSON log path updated after alignment. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `dask` | `False` | Standard local execution setting for this notebook. |
| `batch_size_dask` | `30` | Batch size paired with `dask`; keep the default value for the standard local workflow. |
| `suffix` | `None` | Optional label appended to the aligned output filename as `suffix-<suffix>`. |

Before alignment, check that each subject has exactly one selected source file and exactly one selected transform file for each structure.

**Cortical**

Apply cortical transformations estimated from `run="01"` to cortical response data from `run="02"`.


In [ ]:
for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="response",
        xform_structure_name=None,
        xform_name_filter={"task": "rhaPRcortical"},
        njobs=2,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="rha_align_cortical",
        suffix="run02",
    )


**Subcortical**

Apply subcortical transformations estimated from `run="01"` to subcortical response data from `run="02"`.


In [ ]:
for structure in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="response",
        xform_structure_name=None,
        xform_name_filter={"task": "rhaPRsubcortical"},
        njobs=2,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="rha_align_subcortical",
        suffix="run02",
    )


**Volume ROI**

Apply ROI transformations estimated from `run="01"` to ROI response data from `run="02"`.


In [ ]:
ha.align_pipe(
    work_dir,
    sub_list,
    source_step="resample",
    source_modality="response",
    source_structure_name=f"volume-{roi_name}",
    source_name_filter=align_source_filter,
    xform_modality="response",
    xform_structure_name=None,
    xform_name_filter={"task": "rhaPRroi"},
    njobs=2,
    verbose=1,
    dtype=np.float32,
    json_path=json_path,
    overwrite=False,
    log_num="rha_align_roi",
    suffix="run02",
)


## GUI Pipeline

Launch the graphical interface with `gui()`. The user completes the RHA workflow by setting fields, selecting structures, saving or loading configurations as needed, and clicking `RUN!` inside the GUI.


In [ ]:
from fmriha.gui import gui

gui()


### (1) Searchlight Preparation

The GUI prepares searchlights from the values in `Work Dir`, `n_jobs Parameters`, and `Space and Searchlight Configuration`. These values are used automatically by the enabled RHA stage panels when `RUN!` is clicked.

**Screenshot placeholder:** Main GUI after launching `gui()`, with `Work Dir` filled in and the sidebar `n_jobs Parameters` button visible.

**Screenshot placeholder:** `Space and Searchlight Configuration` window showing the Surface and Volume sections.

**Cortical**

Cortical searchlights are generated when a cortical structure is selected in `Template Calculation` or `Transformation Matrix Calculation`.

| GUI field | Default | Internal parameter or value | Meaning |
|---|---|---|---|
| `Space and Searchlight Configuration -> Surface -> Space` | `onavg-ico32` | Surface searchlight setting | Surface space for `hemi-L` and `hemi-R`. |
| `Space and Searchlight Configuration -> Surface -> Surface Searchlight Radius` | `20` | Surface searchlight setting | Searchlight radius in millimeters. |
| Cortical structure selection | Empty until selected | Selected hemisphere | `hemi-L` maps to left hemisphere; `hemi-R` maps to right hemisphere. |
| Automatic cortical mask | `True` | Automatic cortical setting | Searchlights are generated in masked cortical vertex indexing. |
| Automatic distance output | `True` | Automatic cortical setting | Distances are generated for weighted aggregation. |
| `n_jobs Parameters -> Template Calculation` | Half of host CPU count, minimum 1 | `njobs` in `ha.cspace_sls` | Worker count for common-space construction. |
| `n_jobs Parameters -> Transformation Matrix Calculation` | Half of host CPU count, minimum 1 | `njobs` in `ha.xform_sls_con` | Worker count for transformation matrix calculation. |

**Subcortical**

Subcortical searchlights are generated from a CIFTI reference when subcortical structures are selected.

| GUI field | Default | Internal parameter or value | Meaning |
|---|---|---|---|
| `Space and Searchlight Configuration -> Volume -> CIFTI Reference` | Empty | `cifti_file` in `gensls.downsample_cifti_subcortical` | Reference CIFTI file used to derive subcortical masks. |
| Subcortical structure selection | Empty until selected | returned mask key | `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM` map to `l`, `r`, and `brain_stem`. |
| Automatic dense mode | Fixed by GUI | `N=None`, `sls_type="seed"` | Keeps all available subcortical voxels as searchlight centers. |
| `Space and Searchlight Configuration -> Volume -> Volume Searchlight Radius` | `4` | `radius` in `gensls.generate_searchlight_vol` | Searchlight radius in voxel units. |
| `Space and Searchlight Configuration -> Volume -> Threshold` | `0.2` | `threshold` in `gensls.generate_searchlight_vol` | Minimum in-mask fraction for keeping a volume searchlight. |
| `n_jobs Parameters -> Data Preparation` | Minimum of 5 and half of host CPU count | `njobs` in `gensls.generate_searchlight_vol` | Worker count used while generating volume searchlights. |
| Automatic cache | Created on first use | `work_dir / "masks" / "sls_dists_vol.pkl"` | Reused searchlight and distance file for CIFTI subcortical stages. |

**Volume ROI**

Volume ROI searchlights are generated from a NIfTI mask and saved with an ROI-specific cache name.

| GUI field | Default | Internal parameter or value | Meaning |
|---|---|---|---|
| `Space and Searchlight Configuration -> Volume -> NIFTI Mask` | Empty | `mask_dense` in `gensls.generate_searchlight_vol` | NIfTI mask whose non-zero voxels define the ROI. |
| `Space and Searchlight Configuration -> Volume -> NIFTI ROI Name` | Empty | `roi_name` | ROI label used in folder names such as `volume-mPFC`. |
| `Space and Searchlight Configuration -> Volume -> Volume Searchlight Radius` | `4` | `radius` in `gensls.generate_searchlight_vol` | Searchlight radius in voxel units. |
| `Space and Searchlight Configuration -> Volume -> Threshold` | `0.2` | `threshold` in `gensls.generate_searchlight_vol` | Minimum in-mask fraction for keeping an ROI searchlight. |
| `n_jobs Parameters -> Data Preparation` | Minimum of 5 and half of host CPU count | `njobs` in `gensls.generate_searchlight_vol` | Worker count used while generating ROI searchlights. |
| Automatic cache | Created on first use | `work_dir / "masks" / "sls_dists_vol_<roi_name>.pkl"` | Reused searchlight and distance file for ROI stages. |


### (2) Common Space Construction

In the GUI, common-space construction is configured in the `Template Calculation` panel. When the panel is checked and `RUN!` is clicked, the selected settings are passed to `ha.cspace_sls`.

**Screenshot placeholder:** `Template Calculation` tab with the check box enabled, structures selected, task name entered, file filters filled, and subject list visible.

**Field mapping for `ha.cspace_sls`**

| Function parameter | Function default | GUI source | Meaning |
|---|---|---|---|
| `work_dir` | Required | `Work Dir` | Root HA work directory. |
| `sls` | Required | Generated from selected structures and searchlight configuration | Searchlight indices for each selected structure. |
| `dists` | Required | Generated with the searchlights | Distance arrays for weighted aggregation. |
| `radius` | Required | Surface or volume searchlight radius | Radius paired with the selected structure type. |
| `sub_list` | Required | `Template Calculation -> Sub List` or Data Preparation output | Subjects included in common-space construction. |
| `task_name` | `"rha1"` | `Template Calculation -> Task Name` | Task label written into common-space filenames. |
| `cspace_kind` | `"procrustes"` | `Template Calculation -> Common Space Kind` | Common-space algorithm, such as `procrustes` or `pca`. |
| `common_topography` | `False` | `Template Calculation -> Common Topography` | Topographic reconstruction setting for PCA-like common spaces. |
| `weight` | `True` | `Template Calculation -> Weight` | Distance-weighted searchlight aggregation. |
| `demean` | `True` | `Template Calculation -> Demean` | Demeans data before common-template computation. |
| `start_sub` | `0` | `Template Calculation -> Start Sub` | Reference subject index. |
| `chunk_size` | `2000` | `Template Calculation -> Chunk Size` | Number of searchlights per chunk. |
| `njobs` | `5` | `n_jobs Parameters -> Template Calculation` | Number of joblib workers. |
| `step` | `"resample"` | `Template Calculation -> Step` | Source step folder. |
| `modality` | `"response"` | `Template Calculation -> Modality` | Source modality. |
| `structure_name` | `"hemi-L"` | `Template Calculation -> Structure` | Selected structures converted to folder names. |
| `dtype` | `np.float32` | `Template Calculation -> Float Type` | `np.float32` or `np.float64`. |
| `scope` | `"sls"` | `Template Calculation -> Scope` | Scope label written into filenames. |
| `json_path` | `"prep_log.json"` | `Work Dir / logs / prep_log.json` | Processing-record JSON path. |
| `overwrite` | `False` | Automatic value | JSON record behavior during GUI runs. |
| `log_num` | `"00001"` | Automatic value `cspace` | Progress log suffix. |
| `dask` | `False` | Automatic value | Standard GUI execution setting. |
| `cluster` | `None` | Automatic value | Standard GUI execution setting. |
| `**file_filters` | Empty | `Template Calculation -> File Filter` plus `Suffix` when filled | BIDS-like filters for selecting prepared input files. |

**Cortical**

Select `hemi-L`, `hemi-R`, or both in `Template Calculation -> Structure`. Use a cortical task name, for example `rhaPRcortical`, and file filters that select the training run, such as `run=01`.

**Subcortical**

Select one or more of `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM`. Fill the CIFTI reference in `Space and Searchlight Configuration` before running this stage.

**Volume ROI**

Select `volume-ROI`. Fill `NIFTI Mask` and `NIFTI ROI Name` in `Space and Searchlight Configuration` before running this stage. The GUI converts `volume-ROI` to `volume-<roi_name>` for the function call.


### (3) Transformation Matrix Calculation

In the GUI, transformation matrix calculation is configured in the `Transformation Matrix Calculation` panel. The panel calls `ha.xform_sls_con` for selected cortical, subcortical, and ROI structures.

**Screenshot placeholder:** `Transformation Matrix Calculation` tab with the check box enabled, structures selected, common-space filters visible, source file filters filled, and subject list visible.

**Field mapping for `ha.xform_sls_con`**

| Function parameter | Function default | GUI source | Meaning |
|---|---|---|---|
| `work_dir` | Required | `Work Dir` | Root HA work directory. |
| `sls` | Required | Generated from selected structures and searchlight configuration | Searchlight indices for each selected structure. |
| `dists` | Required | Generated with the searchlights | Distance arrays for weighted transform aggregation. |
| `radius` | Required | Surface or volume searchlight radius | Radius paired with the selected structure type. |
| `sub_list` | Required | `Transformation Matrix Calculation -> Sub List` or Data Preparation output | Subjects used for transformation matrix calculation. |
| `source_step` | `"resample"` | `Transformation Matrix Calculation -> Step` | Source step folder. |
| `modality` | `"response"` | `Transformation Matrix Calculation -> Modality` | Source and transform modality. |
| `structure_name` | `["hemi-L", "hemi-R"]` | `Transformation Matrix Calculation -> Structure` | Selected structures converted to folder names. |
| `task_name` | `"rha1"` | `Transformation Matrix Calculation -> Task Name` | Common-space task name to select and transform task name to save. |
| `cspace_desc` | `None` | `Transformation Matrix Calculation -> Common Space` | Key-value filters for selecting the common-space file, such as `alg=procrustes`. |
| `reflection` | `True` | `Transformation Matrix Calculation -> Reflection` | Reflection setting for Procrustes transforms. |
| `scaling` | `False` | `Transformation Matrix Calculation -> Scaling` | Scaling setting for Procrustes transforms. |
| `weight` | `True` | `Transformation Matrix Calculation -> Weight` | Distance-weighted aggregation of local transformation matrices. |
| `chunk_size` | `2000` | `Transformation Matrix Calculation -> Chunk Size` | Number of searchlights per chunk. |
| `check_completeness` | `False` | Automatic value | Compatibility argument retained by the API. |
| `dtype` | `np.float32` | `Transformation Matrix Calculation -> Float Type` | `np.float32` or `np.float64`. |
| `njobs` | `5` | `n_jobs Parameters -> Transformation Matrix Calculation` | Number of joblib workers. |
| `scope` | `"sls"` | `Transformation Matrix Calculation -> Scope` | Scope label written into transform filenames. |
| `json_path` | `"prep_log.json"` | `Work Dir / logs / prep_log.json` | Processing-record JSON path. |
| `overwrite` | `False` | Automatic value | JSON record behavior during GUI runs. |
| `log_num` | `"00001"` | Automatic value `xform` | Progress log suffix. |
| `**file_filters` | Empty | `Transformation Matrix Calculation -> File Filter` plus `Suffix` when filled | BIDS-like filters for selecting source response files. |

**Cortical**

Select `hemi-L`, `hemi-R`, or both. Match `Task Name` to the cortical common-space task, and use source file filters such as `run=01`.

**Subcortical**

Select one or more subcortical structures. Match `Task Name` to the subcortical common-space task, and use common-space filters such as `alg=procrustes`.

**Volume ROI**

Select `volume-ROI`. The GUI uses the ROI name from `Space and Searchlight Configuration` to locate `volume-<roi_name>` folders and ROI searchlight caches.


### (4) Apply Transformation Matrix

In the GUI, applying transformation matrices is configured in the `Alignment` panel. When the panel is checked and `RUN!` is clicked, the selected settings are passed to `ha.align_pipe` for each selected source structure.

**Screenshot placeholder:** `Alignment` tab with source structure selected, source name filters and transform name filters filled, suffix entered, and subject list visible.

**Screenshot placeholder:** Bottom action bar showing `Save Configuration`, `Load Configuration`, `Reset All Configuration`, and `RUN!`, plus the sidebar `Monitor` button.

**Field mapping for `ha.align_pipe`**

| Function parameter | Function default | GUI source | Meaning |
|---|---|---|---|
| `work_dir` | Required | `Work Dir` | Root HA work directory. |
| `sub_list` | Required | `Alignment -> Sub List` or Data Preparation output | Subjects to align. |
| `source_step` | Required | `Alignment -> Source Step` | Step folder containing source data to align. |
| `source_modality` | Required | `Alignment -> Source Modality` | Source modality. |
| `source_structure_name` | Required | `Alignment -> Source Structure` | Selected structure converted to the source folder name. |
| `source_name_filter` | Required | `Alignment -> Source Name Filter` | BIDS-like filters selecting one source `.npy` file per subject, such as `run=02`. |
| `xform_modality` | Required | `Alignment -> Transform Modality` | Transform modality under each subject's `xforms` folder. |
| `xform_structure_name` | Required | Automatic value | The selected source structure is used for transform lookup. |
| `xform_name_filter` | Required | `Alignment -> Transform Name Filter` | BIDS-like filters selecting one transform `.npz` file per subject, such as `task=rhaPRcortical`. |
| `njobs` | `5` | `n_jobs Parameters -> Alignment` | Number of joblib workers. |
| `verbose` | `1` | Automatic value | Joblib verbosity level used by the GUI run. |
| `dtype` | `np.float32` | `Alignment -> Float Type` | `np.float32` or `np.float64`. |
| `json_path` | `"prep_log.json"` | `Work Dir / logs / prep_log.json` | Processing-record JSON path. |
| `overwrite` | `False` | Automatic value | JSON record behavior during GUI runs. |
| `log_num` | `"00001"` | Automatic value `alignment` | Progress log suffix. |
| `dask` | `False` | Automatic value | Standard GUI execution setting. |
| `batch_size_dask` | `30` | Automatic value | Standard GUI execution setting. |
| `suffix` | `None` | `Alignment -> Suffix` | Optional label appended to aligned output filenames. |

**Cortical**

Select `hemi-L`, `hemi-R`, or both. Use source filters for the data being aligned, such as `run=02`, and transform filters matching the cortical transform task, such as `task=rhaPRcortical`.

**Subcortical**

Select one or more subcortical structures. The GUI maps them to `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM` for the alignment call.

**Volume ROI**

Select `volume-ROI`. The GUI maps it to `volume-<roi_name>` using the ROI name from `Space and Searchlight Configuration`. Use transform filters matching the ROI transform task, such as `task=rhaPRroi`.
